# MATH 472 Homework 4
## Holly Bossart
## Due 02/13/2020

## Problem 2.5
There were 46 crude oil spills of at least 1000 barrels from tankers in the US waters during 1974-1999. The website for this book contains the following data: the number of spills in the $i$th year, $N_i$; the estimated amount of oil shipped through US waters as part of US import/export operations in the $i$th year, adjusted for spillage in international or foreign waters, $b_{i1}$; and the amount of oil shipped through US waters during domestic shipments in the $i$th year, $b_{i2}$. The data are adapted. Oil shipment amounts are measured in billions of barrels (Bbbl).

The volume of oil shipped is a measure of exposure to spill risk. Suppose we use the Poisson process assumption given by $N_i|b_{i1}, b_{i2} \sim $ Poisson($\lambda_i$), where $\lambda_i = \alpha_1 b_{i1} + \alpha_2 b_{i2}$. The parameters of this model are $\alpha_1$ and $\alpha_2$, which represent the rate of spill occurrence per Bbbl oil shipped during import/export and domestic shipments, respectively.

### (a) Derive the Newton-Raphson update for finding the MLEs of $\alpha_1$ and $\alpha_2$

### (a) solution

The likelihood function, given that $N_i| \lambda \sim $ Poisson($\lambda_i$) where $\lambda_i = \mathbf{\alpha}^T\mathbf{b_i} =  \alpha_1 b_{i1} + \alpha_2 b_{i2}$ is: 
$$L(\mathbf{\alpha}) = \prod_{i=1}^{n} \frac{1}{N_i!} (\mathbf{\alpha^T b_i})^{N_i} e^{\mathbf{ \alpha^T b_i}} $$

By taking the logarithm, we get the following log-likelihood function:
$$l(\mathbf{\alpha}) = \sum_{i=1}^{n} N_i \log(\mathbf{\alpha^T b_i}) - \sum_{i=1}^{n} \log(N_i!) - \sum_{i=1}^{n} \mathbf{\alpha^T b_i} $$

Then, the first derivative of the log-likelihood is:
$$l'(\mathbf{\alpha}) = \sum_{i=1}^{n} N_i \mathbf{\frac{b_i}{\alpha^T b_i}} - \sum_{i=1}^{n} \mathbf{b_i} $$ 

With the second derivative being:
$$ l''(\mathbf{\alpha}) = \sum_{i=1}^{n} -N_i \mathbf{\frac{b_i b_i^T}{(\alpha^T b_i)^2}} $$

Thus, using the Newton-Raphson method and Eq. 2.33, the updating equation for $\alpha$ is:
$$ \mathbf{\alpha}^{(t+1)} =  \mathbf{\alpha}^{(t)} -\Big[\sum_{i=1}^{n} N_i \mathbf{\frac{b_i}{(\alpha^{(t)T} b_i)}} - \sum_{i=1}^{n} \mathbf{b_i}\Big]^{-1} \sum_{i=1}^{n} -N_i \mathbf{\frac{b_i b_i^T}{(\alpha^{(t)T} b_i)^2}}. $$

### (b) Derive the Fisher scoring update for finding the MLEs of $\alpha_1$ and $\alpha_2$.

### (b) solution

We will use the above log-likelihood and its derivative to utilize the Fisher Information update:

$$\mathbf{\alpha^{(t+1)}} = \mathbf{\alpha^{(t)}} + \mathbf{I(\alpha^{(t)}})^{-1} l'(\mathbf{\alpha^{(t)}}) $$

We know that the Fisher information matrix $I(\mathbf{\alpha}) = -E[l''(\mathbf{\alpha})]$.

By solving for the Fisher information matrix in that way, we get:
$$ I(\mathbf{\alpha}) = -E[l''(\mathbf{\alpha})] = \sum^n_{i=1} \frac{\mathbf{b_i b_i^T}}{\mathbf{(\alpha^T b_i)^2}}.$$

Plugging that into the Fisher Information updating equation, we get:
$$ \mathbf{\alpha^{(t+1)}} = \mathbf{\alpha^{(t)}} + \Big [ \sum^n_{i=1} \frac{\mathbf{b_i b_i^T}}{\mathbf{(\alpha^{(t)T} b_i)^2}} \Big ]^{-1} l'(\mathbf{\alpha^{(t)}})$$

### (c) Implement the Newton-Raphson and Fisher scoring methods for this problem, provide the MLEs of $\alpha_1$ and $\alpha_2$, and compare the implementation ease and performance of the two methods.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from scipy.linalg import inv
import pandas as pd

data = pd.read_csv("oilspills.dat", delimiter = " ")

year = data['year'].to_numpy().reshape((26,1))
N = data['spills'].to_numpy().reshape((26,1))
b = data[['importexport', 'domestic']].to_numpy().reshape((26,2))
b1 = data[['importexport']].to_numpy().reshape((26,1))
b2 = data[['domestic']].to_numpy().reshape((26,1))
n = len(year)

def loglike(a1, a2):
    summation = []
    alpha = np.asarray((a1,a2)).reshape(2,1)
    for i in range(n):
        b_i = b[i].reshape(2,1).T
        summation.append(N[i]*np.log(b_i.dot(alpha))-np.log(np.math.factorial(N[i]))-(b_i.dot(alpha)))
    
    summation=np.asarray(summation)
    summation=summation.sum(axis=0)
    return summation

def llprime(a1, a2):
    summation = []
    alpha = np.asarray((a1,a2)).reshape((2,1))
    for i in range(n):
        Ni = N[i]
        bi = np.asarray(b[i]).reshape((2,1))
        dotprod = np.dot(alpha.T, bi)
        value = (Ni * bi / dotprod) - bi
        summation.append(value)
    summation=np.asarray(summation)
    summation=summation.sum(axis=0)
    return summation


#def llprime2
def llprime2(a1, a2):
    summation = []
    alpha = np.asarray((a1,a2)).reshape((2,1))
    for i in range(n):
        Ni = N[i]
        bi = np.asarray(b[i]).reshape((2,1))
        dotprod = np.dot(alpha.T, bi)**2
        value = (Ni/dotprod)*np.dot(bi, bi.T)
        summation.append(value)
    summation=np.asarray(summation)
    summation=summation.sum(axis=0)
    return -summation


newtons_hess=[]
def newt_step(a1, a2):
    ie = -np.linalg.inv(llprime2(a1,a2))
    newtons_hess.append(ie)
    jj = llprime(a1,a2)
    return np.dot(ie, jj)

def newton_method(a1, a2, max_iterations, print_option):
    num_iterations = 0
    alpha_ones = []
    alpha_twos = []
    alpha_iter = np.asarray((a1,a2)).reshape((2,1))
    
    while num_iterations < max_iterations:
        alpha_ones.append(alpha_iter[0])
        alpha_twos.append(alpha_iter[1])
        alpha_iter = alpha_iter + newt_step(alpha_iter[0],alpha_iter[1]) 
        num_iterations += 1

    if print_option == 0:
        alpha_ones = np.asarray(alpha_ones).reshape(num_iterations, 1)
        alpha_twos = np.asarray(alpha_twos).reshape(num_iterations, 1)
        alphas = np.hstack((alpha_ones, alpha_twos))
        return alphas
    
    if print_option == 1:
        print('Using Newtons method')
        print('Starting values: ', alpha_ones[0], alpha_twos[0])
        print('Approximation values: ', alpha_ones[-1], alpha_twos[-1])
        print('Number of iterations: ', num_iterations)
        
fishers_hess = []      
def fisher(a1, a2):
    summation = []
    alpha = np.asarray((a1,a2)).reshape((2,1))
    for i in range(n):
        b_i = b[i].reshape((2,1))
        summation.append((1/(alpha.T.dot(b_i)))*(b_i.dot(b_i.T)))
    summation=np.asarray(summation)
    summation=summation.sum(axis=0)
    return summation
        

def fisher_method(a1, a2, max_iterations, print_option):
    num_iterations = 0
    alpha_ones = []
    alpha_twos = []
    alpha_iter = np.asarray((a1,a2)).reshape((2,1))
    
    while num_iterations < max_iterations:
        alpha_ones.append(alpha_iter[0])
        alpha_twos.append(alpha_iter[1])
        inv_fish = inv(fisher(alpha_iter[0], alpha_iter[1]))
        fishers_hess.append(inv_fish)
        alpha_iter = alpha_iter + np.dot(inv_fish, llprime(alpha_iter[0], alpha_iter[1]))
        num_iterations += 1

    if print_option == 0:
        alpha_ones = np.asarray(alpha_ones).reshape(num_iterations, 1)
        alpha_twos = np.asarray(alpha_twos).reshape(num_iterations, 1)
        alphas = np.hstack((alpha_ones, alpha_twos))
        return alphas
    
    if print_option == 1:
        print('\nUsing Fishers method')
        print('Starting values: ', alpha_ones[0], alpha_twos[0])
        print('Approximation values: ', alpha_ones[-1], alpha_twos[-1])
        print('Number of iterations: ', num_iterations)
        print()
    
    
    

a1 = 0.5
a2 = 0.5
newts = newton_method(a1, a2, 8, 0)
newtons_hess = np.asarray(newtons_hess)
fish = fisher_method(a1, a2, 10, 0)
fishers_hess = np.asarray(fishers_hess)

### (c) solution

#### Newton's Method Approximations to $\mathbf{\alpha}$

| Iteration, $t$ | $\alpha^{(t)}$                                          | $-l''(\alpha^{(t+1)})$                                                                              |
|--------------|--------------------------------------------------------|----------------------------------------------------------------------------------------------------|
| 4            | $\begin{pmatrix} 1.08904959\\ 0.9364765 \end{pmatrix}$            | $\begin{bmatrix} 0.14994565 &-0.16614134\\ -0.16614134 & 0.30467123 \end{bmatrix}$                   |
| 5            | $\begin{pmatrix}1.09709646\\ 0.93756821\end{pmatrix}$ | $\begin{bmatrix}   0.15178677& -0.16802504\\   -0.16802504 & 0.30765026  \end{bmatrix}$          |
| 6            | $\begin{pmatrix}1.09715253\\ 0.93755459\end{pmatrix}$ | $\begin{bmatrix}    0.15179824 & -0.16803609\\    -0.16803609 & 0.30766546   \end{bmatrix}$       |
| 7           | $\begin{pmatrix}1.09715253\\ 0.93755458\end{pmatrix}$ | $\begin{bmatrix}     0.15179824 &-0.16803609\\     -0.16803609 & 0.30766546    \end{bmatrix}$      |
| 8            | $\begin{pmatrix}1.09715253\\ 0.93755458\end{pmatrix}$ | $\begin{bmatrix}      0.15179824 & -0.16803609\\      -0.16803609 & 0.30766546      \end{bmatrix}$ |




#### Fisher Approximation to $\mathbf{\alpha}$
| Iteration, $t$ | $\alpha^{(t)}$                                          | $I(\alpha^{(t)})^{-1}$                                                                              |
|--------------|--------------------------------------------------------|----------------------------------------------------------------------------------------------------|
| 6            | $\begin{pmatrix} 1.09735592 \\ 0.93724634 \end{pmatrix}$      | $\begin{bmatrix} 0.1914646&  -0.22814076\\ -0.22814076 & 0.3987394 \end{bmatrix}$                   |
| 7            | $\begin{pmatrix}1.09708834 \\ 0.93765188\end{pmatrix}$ | $\begin{bmatrix}   0.19145227 &-0.2281372\\   -0.2281372 &  0.39875693  \end{bmatrix}$          |
| 8            | $\begin{pmatrix}1.0971728 \\ 0.93752387\end{pmatrix}$ | $\begin{bmatrix}    0.19145616 & -0.22813832\\    -0.22813832 & 0.3987514   \end{bmatrix}$       |
| 9           | $\begin{pmatrix}1.09714614 \\ 0.93756428\end{pmatrix}$ | $\begin{bmatrix}     0.19145493 & -0.22813797\\     -0.22813797 & 0.39875315    \end{bmatrix}$      |
| 10            | $\begin{pmatrix}1.09715455 \\ 0.93755152\end{pmatrix}$ | $\begin{bmatrix}      0.19145532 & -0.22813808\\      -0.22813808 & 0.3987526      \end{bmatrix}$ |


### (d) Estimate standard errors for the MLEs of $\alpha_1$ and $\alpha_2$.

To estimate the standard errors for the MLEs of $\alpha_1$ and $\alpha_2$, I utilize the book's technique of taking the square root of the diagonal components of the chosen estimates for $\mathbf{I(\hat \alpha^*)} ^{-1}$. I use the last five iterations of the above Fisher approximation of $\alpha$ in the table below.

In [2]:
alpha1_se = []
alpha2_se = []
def mle_se():
    
    for i in range(4,10):
        alpha1_se.append(np.sqrt(fishers_hess[i, 0, 0]))
        alpha2_se.append(np.sqrt(fishers_hess[i, 1, 1]))
    #print(alpha1_se)
    #print(alpha2_se)

mle_se()


### (d) solution

The Fisher approximation for the standard errors of the parameters $\alpha$

| Iteration, $t$ | Standard error approximations for $\alpha^{(t)}$                                          | $I(\alpha^{(t)})^{-1}$                                                                              |
|--------------|--------------------------------------------------------|----------------------------------------------------------------------------------------------------|
| 6            | $\begin{pmatrix} 0.43752204646404347 \\ 0.631502105030092 \end{pmatrix}$      | $\begin{bmatrix} 0.1914646&  -0.22814076\\ -0.22814076 & 0.3987394 \end{bmatrix}$                   |
| 7            | $\begin{pmatrix}0.4375666797712071 \\ 0.6314581552315659\end{pmatrix}$ | $\begin{bmatrix}   0.19145227 &-0.2281372\\   -0.2281372 &  0.39875693  \end{bmatrix}$          |
| 8            | $\begin{pmatrix}0.4375525918458805 \\ 0.631472037645374\end{pmatrix}$ | $\begin{bmatrix}    0.19145616 & -0.22813832\\    -0.22813832 & 0.3987514   \end{bmatrix}$       |
| 9           | $\begin{pmatrix}0.4375570390385862 \\ 0.6314676563471344\end{pmatrix}$ | $\begin{bmatrix}     0.19145493 & -0.22813797\\     -0.22813797 & 0.39875315    \end{bmatrix}$      |
| 10            | $\begin{pmatrix}0.43755607836279314 \\ 0.6314686028681602\end{pmatrix}$ | $\begin{bmatrix}      0.19145532 & -0.22813808\\      -0.22813808 & 0.3987526      \end{bmatrix}$ |

### (e) Apply the method of steepest ascent. Use step-halving backtracking as necessary.

In [3]:
def steep_ascent(a1, a2, scale, max_iterations, print_option):
    identity = np.identity(2)
    alpha = np.asarray((a1, a2)).reshape((2,1))
    alpha_ones = [a2]
    alpha_twos = [a2]
    num_iters = 1

    for i in range(max_iterations):
        new = alpha + scale*identity.dot(llprime(alpha[0], alpha[1])) 
        
        if loglike(new[0], new[1]) > loglike(alpha[0], alpha[1]):
            alpha = new
            alpha_ones.append(alpha[0])
            alpha_twos.append(alpha[1])
            num_iters += 1
            
        else:
            scale = scale / 2.0
            
    if print_option == 0:
        alpha_ones = np.asarray(alpha_ones).reshape(num_iters, 1)
        alpha_twos = np.asarray(alpha_twos).reshape(num_iters, 1)
        alphas = np.hstack((alpha_ones, alpha_twos))
        return alphas
    
    if print_option == 1:
        #print('\nUsing steepest ascent method')
       # print('Starting values: ', alpha_ones[0], alpha_twos[0])
        #print('Approximation values: ', alpha_ones[-5:], alpha_twos[-5:])
        #print()
    
    
steep_ascent(a1, a2, 1, 100, 1)

IndentationError: expected an indented block (<ipython-input-3-c7ce5584ca81>, line 33)

### (e) solution

Using method of steepest ascent with 100 iterations to estimate $\mathbf\alpha$.

| Iteration, t | $\alpha^{(t)}$                                         |
|--------------|-------------------------------------------------------|
| 96           | $\begin{pmatrix} 1.09715291\\ 0.937554 \end{pmatrix}$ |
| 97           | $\begin{pmatrix}1.09715285\\0.93755409\end{pmatrix}$  |
| 98           | $\begin{pmatrix}1.0971528\\0.93755416\end{pmatrix}$   |
| 99           | $\begin{pmatrix}1.09715276\\0.93755423\end{pmatrix}$  |
| 100          | $\begin{pmatrix}1.09715273\\0.93755428\end{pmatrix}$  |

### (f) Apply quasi-Newton optimization with the Hessian approximation update given in (2.49). Compare performance with and without step-halving.

In [ ]:
m = -fisher(0.5, 0.5)
def update_hessian(alpha0, alpha1, m, max_iterations):
    num_iter = 0
    alph_list = [alpha0, alpha1]
    
    while num_iter < max_iterations:
        z = alpha1 - alpha0
        y = llprime(alpha1[0], alpha1[1]) - llprime(alpha0[0], alpha0[1])
        v = y - np.dot(m, z)
        denom = np.dot(v.T, z)  
        
        if ((denom>0) | (denom == 0)):
            m = m

        else:
            c = 1/denom
            m = m + c*np.dot(v, v.T)
               
        temp = alpha1
        alpha1 = alpha0 - np.dot(inv(m), llprime(alpha0[0], alpha0[1]))
        alph_list.append(alpha1)
        alpha0 = temp
        num_iter += 1
        
    return alph_list

startalpha = np.asarray([0.5, 0.5]).reshape((2,1))
# update_hessian(startalpha,startalpha, m, 30) this will print everything 

### (f) solution

Using 30 iterations of the Quasi-Newton method with a starting $M^{(0)} = -I(\alpha)$, the results are as follows:

| Iteration, t | $\alpha^{(t)}$                                          |
|--------------|---------------------------------------------------------|
| 26           | $\begin{pmatrix} 1.09715713\\ 0.93754862 \end{pmatrix}$ |
| 27           | $\begin{pmatrix}1.09715713\\0.93754862\end{pmatrix}$    |
| 28           | $\begin{pmatrix}1.09715494\\0.93755149\end{pmatrix}$    |
| 29           | $\begin{pmatrix}1.09715494\\0.93755149\end{pmatrix}$    |
| 30           | $\begin{pmatrix}1.0971538\\0.93755298\end{pmatrix}$     |

### (g) Construct a graph resembling Figure 2.8 that compares the paths taken by method used in (a)-(f). Choose the plotting region and starting point to best illustrate the features of the algorithms' performance.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
alpha_0_plot = np.linspace(0, 3, 100)
alpha_1_plot = np.linspace(0, 3, 100)
# make a meshgrid
a_0, a_1 = np.meshgrid(alpha_0_plot, alpha_1_plot)

Zmesh = np.zeros((a_0.shape[0],a_1.shape[0]))
alphaplot = np.asarray([0.5,0.5]).reshape((2,1))

def likeli(N,b,alpha_1,alpha_2):
    total = []
    alpha = np.array([alpha_1,alpha_2]).reshape(2,1)
    for i in range(n):
        b_i = b[i].reshape((2,1))
        total.append(N[i]*np.log(alpha.T.dot(b_i)) - np.log(np.math.factorial(N[i])) - (alpha.T.dot(b_i)))
    total = np.asarray(total)
    summation = total.sum(axis=0)
    return summation

for i in range(a_0.shape[0]):
    for j in range(a_1.shape[0]):
        a0 = a_0[i,j]; a1 = a_1[i,j]
        Zmesh[i,j]=likeli(N,b,a0,a1)
        
plt.contourf(a_0,a_1,Zmesh, 40, cmap='ocean')
plt.colorbar()
plt.plot(newts[:,0],newts[:,1],'k-')
plt.plot(fish[:,0],fish[:,1],'r-')
plt.title('Contour plot of the log-likelihood function')
plt.xlabel('alpha1')
plt.ylabel('alpha2')